In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

def train_test_split(number_of_elements, testing_size):
    training_index = [i for i in range(0, number_of_elements)]
    testing_index = []
    for i in range(1, testing_size + 1):
        choice = np.random.choice(training_index)
        training_index.remove(choice)
        testing_index.append(choice)
    return training_index, testing_index

def kfold_cross_validation(number_of_folds, data_size):
    lists = []
    for i in range(0, number_of_folds):
        lists.append([])
    for j in range(0, data_size):
        list_index = j % number_of_folds
        lists[list_index].append(j)
    return lists

In [ ]:
class Regression_Neural_Network():
    def __init__(self, epochs=1000, learning_rate=0.001):
        self.w1 = np.array([])
        self.b1 = np.array([])
        self.w2 = np.array([])
        self.b2 = np.array([])
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.beta_1 = 0.9
        self.beta_2 = 0.999

    def fit(self, X, y, hidden_size):
        self.w1 = np.random.randn(X.shape[1], hidden_size) * np.sqrt(2 / X.shape[1])
        self.b1 = np.random.randn(1, hidden_size) * np.sqrt(2 / hidden_size)
        self.w2 = np.random.randn(hidden_size, 1)
        self.b2 = np.random.randn(1, 1)
        first_moment_w1 = np.zeros_like(self.w1)
        second_moment_w1 = np.zeros_like(self.w1)
        first_moment_b1 = np.zeros_like(self.b1)
        second_moment_b1 = np.zeros_like(self.b1)
        first_moment_w2 = np.zeros_like(self.w2)
        second_moment_w2 = np.zeros_like(self.w2)
        first_moment_b2 = np.zeros_like(self.b2)
        second_moment_b2 = np.zeros_like(self.b2)

        for i in range(0, self.epochs):
            z1 = X @ self.w1 + self.b1
            a1 = np.where(z1>0, z1, 0.01*z1)
            z2 = a1 @ self.w2 + self.b2
            a2 = z2
            y_hat = a2
            
            grad_w2 = (np.transpose(a1) @ (-2 * (y-y_hat))) / X.shape[0]
            grad_b2 = (np.sum(-2 * (y-y_hat), axis=0, keepdims=True)) / X.shape[0]
            grad_w1 = (np.transpose(X) @ ((-2 * (y-y_hat)) @ np.transpose(self.w2) * np.where(z1>0, 1, 0.01))) / X.shape[0]
            grad_b1 = (np.sum((-2 * (y-y_hat)) @ np.transpose(self.w2) * np.where(z1>0, 1, 0.01), axis=0, keepdims=True)) / X.shape[0]

            first_moment_w2, second_moment_w2, m_hat_w2, v_hat_w2 = self.adam_optimizing(grad_w2, first_moment_w2, second_moment_w2, i)
            first_moment_b2, second_moment_b2, m_hat_b2, v_hat_b2 = self.adam_optimizing(grad_b2, first_moment_b2, second_moment_b2, i)
            first_moment_w1, second_moment_w1, m_hat_w1, v_hat_w1 = self.adam_optimizing(grad_w1, first_moment_w1, second_moment_w1, i)
            first_moment_b1, second_moment_b1, m_hat_b1, v_hat_b1 = self.adam_optimizing(grad_b1, first_moment_b1, second_moment_b1, i)

            self.w2 = self.w2 - self.learning_rate * (m_hat_w2 / (np.sqrt(v_hat_w2) + 1e-8))
            self.b2 = self.b2 - self.learning_rate * (m_hat_b2 / (np.sqrt(v_hat_b2) + 1e-8))
            self.w1 = self.w1 -self.learning_rate * (m_hat_w1 / (np.sqrt(v_hat_w1) + 1e-8))
            self.b1 = self.b1 - self.learning_rate * (m_hat_b1 / (np.sqrt(v_hat_b1) + 1e-8))

    def adam_optimizing(self, grad, first_moment, second_moment, index):
        first_moment = self.beta_1 * first_moment + (1 - self.beta_1) * grad 
        second_moment = self.beta_2 * second_moment + (1 - self.beta_2) * np.square(grad)
        m_hat = first_moment / (1 - self.beta_1 ** (index+1))
        v_hat = second_moment / (1 - self.beta_2 ** (index+1))
        return first_moment, second_moment, m_hat, v_hat
    
    def predict(self, X):
        z1 = X @ self.w1 + self.b1
        a1 = np.where(z1>0, z1, 0.01*z1)
        z2 = a1 @ self.w2 + self.b2
        a2 = z2
        y_hat = a2
        return y_hat

class Pytorch_Regression_Neural_Network(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, X):
        return self.sequential(X)

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
X = df.drop(columns=['salary_year_avg'])
y = df['salary_year_avg'].to_numpy().reshape(-1, 1)
y = (y - y.mean()) / y.std()
numerical_columns = ['job_posted_month_sine', 'job_posted_year']
numerical_columns = [X.columns.get_loc(x) for x in numerical_columns]
X = X.to_numpy()

r2_values = []
r2_values_pytorch = []
for item in kfold_cross_validation(5, y.shape[0]):
    for column in numerical_columns:
        X[:, column] = (X[:, column] - np.delete(X, item, axis=0)[:, column].mean()) / np.delete(X, item, axis=0)[:, column].std()
    regression_neural_network = Regression_Neural_Network(epochs=2500)
    regression_neural_network.fit(np.delete(X, item, axis=0), np.delete(y, item, axis=0), 50)
    predictions = regression_neural_network.predict(X[item])
    r2 = 1 - (np.sum((y[item] - predictions)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values.append(r2)
    X_testing_tensor = torch.tensor(X[item], dtype=torch.float32)
    y_testing_tensor = torch.tensor(y[item], dtype=torch.float32)
    X_tensor = torch.tensor(np.delete(X, item, axis=0), dtype=torch.float32)
    y_tensor = torch.tensor(np.delete(y, item, axis=0), dtype=torch.float32)
    model = Pytorch_Regression_Neural_Network(X.shape[1], 50)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    model.train()
    for epoch in range(0, 2500):
        y_hat = model(X_tensor)
        loss = criterion(y_hat, y_tensor)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    model.eval()
    with torch.no_grad():
        predictions_pytorch = model(X_testing_tensor)
    r2_pytorch = 1 - (torch.sum((y_testing_tensor - predictions_pytorch)**2)) / (torch.sum((y_testing_tensor - torch.mean(y_testing_tensor))**2))
    r2_values_pytorch.append(r2_pytorch)
print(f'The Average R Squared Score For A Regression Neural Network Is {np.mean(r2_values):.4f}.')
print(f'The Average R Squared Score For Pytorch\'s Regression Neural Network Is {np.mean(r2_values_pytorch):.4f}.')

The Average R Squared Score For A Regression Neural Network Is 0.3563.
The Average R Squared Score For Pytorch's Regression Neural Network Is 0.3292.


In [4]:
class Classification_Neural_Network():
    def __init__(self, epochs=1000, learning_rate=0.001):
        self.w1 = np.array([])
        self.b1 = np.array([])
        self.w2 = np.array([])
        self.b2 = np.array([])
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.beta_1 = 0.9
        self.beta_2 = 0.999

    def fit(self, X, y, hidden_size):
        self.w1 = np.random.randn(X.shape[1], hidden_size) * np.sqrt(2 / X.shape[1])
        self.b1 = np.random.randn(1, hidden_size) * np.sqrt(2 / hidden_size)
        self.w2 = np.random.randn(hidden_size, 1)
        self.b2 = np.random.randn(1, 1)
        first_moment_w1 = np.zeros_like(self.w1)
        second_moment_w1 = np.zeros_like(self.w1)
        first_moment_b1 = np.zeros_like(self.b1)
        second_moment_b1 = np.zeros_like(self.b1)
        first_moment_w2 = np.zeros_like(self.w2)
        second_moment_w2 = np.zeros_like(self.w2)
        first_moment_b2 = np.zeros_like(self.b2)
        second_moment_b2 = np.zeros_like(self.b2)

        for i in range(0, self.epochs):
            z1 = X @ self.w1 + self.b1
            a1 = 1 / (1 + np.exp(-z1))
            z2 = a1 @ self.w2 + self.b2
            a2 = 1 / (1 + np.exp(-z2))
            y_hat = a2
            
            grad_w2 = (np.transpose(a1) @ -(y-y_hat)) / X.shape[0] 
            grad_b2 = (np.sum(-(y-y_hat), axis=0, keepdims=True)) / X.shape[0]
            grad_w1 = (np.transpose(X) @ ((-(y-y_hat) @ np.transpose(self.w2)) * a1 * (np.ones_like(a1)-a1))) / X.shape[0]
            grad_b1 = (np.sum(-(y-y_hat) @ np.transpose(self.w2) * a1 * (np.ones_like(a1)-a1), axis=0, keepdims=True)) / X.shape[0]

            first_moment_w2, second_moment_w2, m_hat_w2, v_hat_w2 = self.adam_optimizing(grad_w2, first_moment_w2, second_moment_w2, i)
            first_moment_b2, second_moment_b2, m_hat_b2, v_hat_b2 = self.adam_optimizing(grad_b2, first_moment_b2, second_moment_b2, i)
            first_moment_w1, second_moment_w1, m_hat_w1, v_hat_w1 = self.adam_optimizing(grad_w1, first_moment_w1, second_moment_w1, i)
            first_moment_b1, second_moment_b1, m_hat_b1, v_hat_b1 = self.adam_optimizing(grad_b1, first_moment_b1, second_moment_b1, i)

            self.w2 = self.w2 - self.learning_rate * (m_hat_w2 / (np.sqrt(v_hat_w2) + 1e-8))
            self.b2 = self.b2 - self.learning_rate * (m_hat_b2 / (np.sqrt(v_hat_b2) + 1e-8))
            self.w1 = self.w1 -self.learning_rate * (m_hat_w1 / (np.sqrt(v_hat_w1) + 1e-8))
            self.b1 = self.b1 - self.learning_rate * (m_hat_b1 / (np.sqrt(v_hat_b1) + 1e-8))

    def adam_optimizing(self, grad, first_moment, second_moment, index):
        first_moment = self.beta_1 * first_moment + (1 - self.beta_1) * grad 
        second_moment = self.beta_2 * second_moment + (1 - self.beta_2) * np.square(grad)
        m_hat = first_moment / (1 - self.beta_1 ** (index+1))
        v_hat = second_moment / (1 - self.beta_2 ** (index+1))
        return first_moment, second_moment, m_hat, v_hat
    
    def predict(self, X):
        z1 = X @ self.w1 + self.b1
        a1 = 1 / (1 + np.exp(-z1))
        z2 = a1 @ self.w2 + self.b2
        a2 = 1 / (1 + np.exp(-z2))
        y_hat = a2
        return y_hat

class Pytorch_Classification_Neural_Network(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, X):
        return self.sequential(X)

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
X = df.drop(columns=['salary_year_avg'])
y = np.where(df['salary_year_avg'] > df['salary_year_avg'].median(), 1, 0).reshape(-1, 1)
numerical_columns = ['job_posted_month_sine', 'job_posted_year']
numerical_columns = [X.columns.get_loc(x) for x in numerical_columns]
X = X.to_numpy()

accuracy_values = []
accuracy_values_pytorch = []
for item in kfold_cross_validation(5, y.shape[0]):
    for column in numerical_columns:
        X[:, column] = (X[:, column] - np.delete(X, item, axis=0)[column].mean()) / np.delete(X, item, axis=0)[column].std()
    total = y[item].shape[0]
    classification_neural_network = Classification_Neural_Network(epochs=2500)
    classification_neural_network.fit(np.delete(X, item, axis=0), np.delete(y, item, axis=0), hidden_size=50)
    predictions = classification_neural_network.predict(X[item])
    predictions = [1 if x > 0.5 else 0 for x in predictions]
    correct = 0
    for predicted, true in zip(predictions, y[item]):
        if predicted == true:
            correct += 1
    accuracy = correct / total
    accuracy_values.append(accuracy)
    X_testing_tensor = torch.tensor(X[item], dtype=torch.float32)
    y_testing_tensor = torch.tensor(y[item], dtype=torch.float32)
    X_tensor = torch.tensor(np.delete(X, item, axis=0), dtype=torch.float32)
    y_tensor = torch.tensor(np.delete(y, item, axis=0), dtype=torch.float32)
    model = Pytorch_Classification_Neural_Network(X.shape[1], 50)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    model.train()
    for epoch in range(0, 2500):
        y_hat = model(X_tensor)
        loss = criterion(y_hat, y_tensor)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    model.eval()
    with torch.no_grad():
        predictions_pytorch = torch.sigmoid(model(X_testing_tensor))
    predictions_pytorch = [1 if x > 0.5 else 0 for x in predictions_pytorch]
    correct_pytorch = 0
    for predicted, true in zip(predictions, y[item]):
        if predicted == true:
            correct_pytorch += 1
    accuracy_pytorch = correct_pytorch / total
    accuracy_values_pytorch.append(accuracy)
print(f'The Average Accuracy For A Classification Neural Network Is {np.mean(accuracy_values):.4%}.')
print(f'The Average Accuracy For Pytorch\'s Classification Neural Network Is {np.mean(accuracy_values_pytorch):.4%}.')

The Average Accuracy For A Classification Neural Network Is 76.4303%.
The Average Accuracy For Pytorch's Classification Neural Network Is 76.4303%.
